In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline

import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from helpers import config

TEST_SAMPLE_SIZE = 0


FEATURES_LOOSE = ["poa", "module_temp"]
FEATURES_STRICT = ["poa_rc", "module_temp"]

In [19]:
def _build_hgbr_pipeline():
    """
    Pipeline with HistGradientBoostingRegressor.
    No selector, no imputer.
    """
    pipe = Pipeline(steps=[
        ("model", HistGradientBoostingRegressor(
            early_stopping=True,
            n_iter_no_change=10,
            random_state=1,
            learning_rate=0.1,
            max_iter=400,
            max_leaf_nodes=32,
            warm_start=False,  # fine-tuning below refits on a different dataset
            verbose=False
        ))
    ])
    return pipe


def _ensure_features_loose(df: pd.DataFrame):
    missing = [c for c in FEATURES_LOOSE if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    return df

def _ensure_features_strict(df: pd.DataFrame):
    missing = [c for c in FEATURES_STRICT if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required features: {missing}")
    return df


def _year_offset(start: pd.Timestamp, years: int) -> pd.Timestamp:
    return start + pd.DateOffset(years=years)


def estimate_hgbr(data, rated_power, system_name, scenario,
                  power_col_name="power", training_years=0,
                  read_model_path="", write_model_path=""):
    """
    Train, evaluate, and optionally fine-tune HGBR regressors for multiple systems
    using a Pipeline(scaler + model). No selector/imputer used.

    Returns:
    - data_list updated with:
        - "GBM_general": predictions from the general model (absolute power)
        - "GBM_fit": predictions from the fine-tuned model (absolute power) after the fine window
    """

    scaled_power_col_name = f"scaled_{power_col_name}"

    # Precompute normalized target and poa_rc if needed
    print(data.columns)

    if scaled_power_col_name not in data.columns:
        data[scaled_power_col_name] = data[power_col_name] / rated_power
    if "poa_rc" not in data.columns and "poa_comp_rc" in data.columns:
        data["poa_rc"] = data["poa_comp_rc"]
    if "poa" not in data.columns and "poa_comp" in data.columns:
         data["poa"] = data["poa_comp"]

    # The original code processes system 0; keep that behavior for parity
    print(f"\nProcessing system: {system_name}")

    train_start = data.index.min()
    train_end = _year_offset(train_start, training_years)

    train_data = data[(data.index >= train_start) & (data.index < train_end)]
    test_data = data[(data.index >= train_end)]

    # Guard on index
    if not isinstance(test_data.index, pd.DatetimeIndex):
        raise TypeError("test_data.index must be a DatetimeIndex")
    test_data = test_data.sort_index()
    train_data = train_data.sort_index()

    # Ensure features exist
    if (scenario == "scenario_1") | (scenario == "scenario_2"):
        _ensure_features_loose(test_data)
        _ensure_features_loose(train_data)
        X_train = train_data[FEATURES_LOOSE]
    elif (scenario == "scenario_3") | (scenario == "scenario_4"):
        _ensure_features_strict(test_data)
        _ensure_features_strict(train_data)
        X_train = train_data[FEATURES_STRICT]

    y_train = train_data[scaled_power_col_name]

    if TEST_SAMPLE_SIZE != 0:
        print(f"TESTING WITH SAMPLE SIZE {TEST_SAMPLE_SIZE}")
        rng = np.random.RandomState(1)
        idx = rng.choice(len(X_train), size=min(TEST_SAMPLE_SIZE, len(X_train)), replace=False)
        X_train = X_train.iloc[idx]
        y_train = y_train.iloc[idx]

    # Build pipeline
    pipe = _build_hgbr_pipeline()

    print("Fitting pipeline...")
    pipe.fit(X_train, y_train)

    # Save general model if requested
    if write_model_path:
        pipeline_filename = f"{write_model_path}/hgbr_pipeline_{system_name}_{scenario}.pkl"
        print(f"Saving pipeline object to: {pipeline_filename}")

        joblib.dump(pipe, pipeline_filename)

    
    # --- Create output column with NaNs ---
    data["gbm_fit"] = np.nan


    
# --- Predict only on the post-training part and align by index ---
    if not test_data.empty:
        if (scenario == "scenario_1") | (scenario == "scenario_2"):
                X_test = test_data[FEATURES_LOOSE]
        elif (scenario == "scenario_3") | (scenario == "scenario_4"):
                X_test = test_data[FEATURES_STRICT]
        pred_norm = pipe.predict(X_test)
        data.loc[X_test.index, "gbm_fit"] = pred_norm * rated_power
    else:
        print("Warning: test_data is empty (no samples after training window).")


    return data

In [20]:
def find_project_root(start: Path = None) -> Path:
    """
    Walk upwards until we find a folder that contains 'data'.
    """
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError("Could not find project root containing 'data'.")

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"

ROOT, DATA_DIR, MODELS_DIR

def pick_system_file(system_tag: str, folder: str, plr=None) -> Path:
    folder_path = DATA_DIR / folder
    matches = sorted(folder_path.glob(f"*{system_tag}*.csv"))

    if plr is not None:
        matches = [m for m in matches if f"PLR-{plr}" in str(m)]

    if not matches:
        raise FileNotFoundError(f"No CSV found for system tag '{system_tag}' in {folder_path}")
    return matches[0]

def set_utctime_index(df: pd.DataFrame, col: str = "utctime") -> pd.DataFrame:
    """
    Ensure `col` exists, parse to datetime, set as index, sort.
    Works even if the index is already datetime.
    """
    df = df.copy()

    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
        df = df.set_index(col)
    else:
        # If col isn't in columns, try converting the existing index
        df.index = pd.to_datetime(df.index, errors="coerce", utc=True)

    # Drop invalid timestamps and sort
    df = df[~df.index.isna()].sort_index()

    return df

In [22]:
scenario = "scenario_1"

helsinki_scenario_1_csv = pick_system_file("Helsinki", folder=f"original/{scenario}")
df_hel_scenario_1 = pd.read_csv(helsinki_scenario_1_csv, sep=",")
df_hel_scenario_1 = set_utctime_index(df_hel_scenario_1)
df_hel_scenario_1 = estimate_hgbr(df_hel_scenario_1, config.rated_power_helsinki*1000, "HEL", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")

kuopio_scenario_1_csv = pick_system_file("Kuopio", folder=f"original/{scenario}")
df_kuo_scenario_1 = pd.read_csv(kuopio_scenario_1_csv, sep=",")
df_kuo_scenario_1 = set_utctime_index(df_kuo_scenario_1)
df_kuo_scenario_1 = estimate_hgbr(df_kuo_scenario_1, config.rated_power_kuopio*1000, "KUO", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


sodankyla20_scenario_1_csv = pick_system_file("Sodankyla_20", folder=f"original/{scenario}")
df_sod20_scenario_1 = pd.read_csv(sodankyla20_scenario_1_csv, sep=",")
df_sod20_scenario_1 = set_utctime_index(df_sod20_scenario_1)
df_sod20_scenario_1 = estimate_hgbr(df_sod20_scenario_1, config.rated_power_sodankyla*1000, "SOT-20", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


sodankyla90_scenario_1_csv = pick_system_file("Sodankyla_90", folder=f"original/{scenario}")
df_sod90_scenario_1 = pd.read_csv(sodankyla90_scenario_1_csv, sep=",")
df_sod90_scenario_1 = set_utctime_index(df_sod90_scenario_1)
df_sod90_scenario_1 = estimate_hgbr(df_sod90_scenario_1, config.rated_power_sodankyla*1000, "SOT-90", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


turku_scenario_1_csv = pick_system_file("KTK", folder=f"original/{scenario}")
df_tku_scenario_1 = pd.read_csv(turku_scenario_1_csv, sep=",")
df_tku_scenario_1 = set_utctime_index(df_tku_scenario_1)
df_tku_scenario_1 = estimate_hgbr(df_tku_scenario_1, config.rated_power_turku*1000, "TKU", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")

Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc',
       'poa_comp', 'poa_comp_rc', 'poa_rc', 'module_temp', 'cell_temp', 'wind',
       'T', 'albedo', 'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: HEL
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_1//hgbr_pipeline_HEL_scenario_1.pkl
Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc',
       'poa_comp', 'poa_comp_rc', 'poa_rc', 'module_temp', 'cell_temp', 'wind',
       'T', 'albedo', 'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: KUO
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_1//hgbr_pipeline_KUO_scenario_1.pkl
Index(['time', 'ghi', 'dhi', 'dni', 'poa_beam', 'poa_diffused', 'poa_ground',
       'poa_beam_rc', 'poa_dif

In [23]:
scenario = "scenario_2"

helsinki_scenario_2_csv = pick_system_file("Helsinki", folder=f"original/{scenario}")
df_hel_scenario_2 = pd.read_csv(helsinki_scenario_2_csv, sep=",")
df_hel_scenario_2 = set_utctime_index(df_hel_scenario_2)
df_hel_scenario_2 = estimate_hgbr(df_hel_scenario_2, config.rated_power_helsinki*1000, "HEL", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


kuopio_scenario_2_csv = pick_system_file("Kuopio", folder=f"original/{scenario}")
df_kuo_scenario_2 = pd.read_csv(kuopio_scenario_2_csv, sep=",")
df_kuo_scenario_2 = set_utctime_index(df_kuo_scenario_2)
df_kuo_scenario_2 = estimate_hgbr(df_kuo_scenario_2, config.rated_power_kuopio*1000, "KUO", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


sodankyla20_scenario_2_csv = pick_system_file("Sodankyla_20", folder=f"original/{scenario}")
df_sod20_scenario_2 = pd.read_csv(sodankyla20_scenario_2_csv, sep=",")
df_sod20_scenario_2 = set_utctime_index(df_sod20_scenario_2)
df_sod20_scenario_2 = estimate_hgbr(df_sod20_scenario_2, config.rated_power_sodankyla*1000, "SOT-20", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


sodankyla90_scenario_2_csv = pick_system_file("Sodankyla_90", folder=f"original/{scenario}")
df_sod90_scenario_2 = pd.read_csv(sodankyla90_scenario_2_csv, sep=",")
df_sod90_scenario_2 = set_utctime_index(df_sod90_scenario_2)
df_sod90_scenario_2 = estimate_hgbr(df_sod90_scenario_2, config.rated_power_sodankyla*1000, "SOT-90", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


turku_scenario_2_csv = pick_system_file("KTK", folder=f"original/{scenario}")
df_tku_scenario_2 = pd.read_csv(turku_scenario_2_csv, sep=",")
df_tku_scenario_2 = set_utctime_index(df_tku_scenario_2)
df_tku_scenario_2 = estimate_hgbr(df_tku_scenario_2, config.rated_power_turku*1000, "TKU", scenario, "power", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")

Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc',
       'poa_comp', 'poa_comp_rc', 'poa_rc', 'module_temp', 'cell_temp', 'wind',
       'T', 'albedo', 'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: HEL
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_2//hgbr_pipeline_HEL_scenario_2.pkl
Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc',
       'poa_comp', 'poa_comp_rc', 'poa_rc', 'module_temp', 'cell_temp', 'wind',
       'T', 'albedo', 'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: KUO
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_2//hgbr_pipeline_KUO_scenario_2.pkl
Index(['time', 'ghi', 'dhi', 'dni', 'poa_beam', 'poa_diffused', 'poa_ground',
       'poa_beam_rc', 'poa_dif

In [24]:
scenario = "scenario_3"

helsinki_scenario_3_csv = pick_system_file("Helsinki", folder=f"original/{scenario}")
df_hel_scenario_3 = pd.read_csv(helsinki_scenario_3_csv, sep=",")
df_hel_scenario_3 = set_utctime_index(df_hel_scenario_3)
df_hel_scenario_3 = estimate_hgbr(df_hel_scenario_3, config.rated_power_helsinki*1000, "HEL", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


kuopio_scenario_3_csv = pick_system_file("Kuopio", folder=f"original/{scenario}")
df_kuo_scenario_3 = pd.read_csv(kuopio_scenario_3_csv, sep=",")
df_kuo_scenario_3 = set_utctime_index(df_kuo_scenario_3)
df_kuo_scenario_3 = estimate_hgbr(df_kuo_scenario_3, config.rated_power_kuopio*1000, "KUO", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


sodankyla20_scenario_3_csv = pick_system_file("Sodankyla_20", folder=f"original/{scenario}")
df_sod20_scenario_3 = pd.read_csv(sodankyla20_scenario_3_csv, sep=",")
df_sod20_scenario_3 = set_utctime_index(df_sod20_scenario_3)
df_sod20_scenario_3 = estimate_hgbr(df_sod20_scenario_3, config.rated_power_sodankyla*1000, "SOT-20", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


sodankyla90_scenario_3_csv = pick_system_file("Sodankyla_90", folder=f"original/{scenario}")
df_sod90_scenario_3 = pd.read_csv(sodankyla90_scenario_3_csv, sep=",")
df_sod90_scenario_3 = set_utctime_index(df_sod90_scenario_3)
df_sod90_scenario_3 = estimate_hgbr(df_sod90_scenario_3, config.rated_power_sodankyla*1000, "SOT-90", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


turku_scenario_3_csv = pick_system_file("KTK", folder=f"original/{scenario}")
df_tku_scenario_3 = pd.read_csv(turku_scenario_3_csv, sep=",")
df_tku_scenario_3 = set_utctime_index(df_tku_scenario_3)
df_tku_scenario_3 = estimate_hgbr(df_tku_scenario_3, config.rated_power_turku*1000, "TKU", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")

Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc',
       'poa_comp', 'poa_comp_rc', 'poa_rc', 'module_temp', 'cell_temp', 'wind',
       'T', 'albedo', 'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: HEL
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_3//hgbr_pipeline_HEL_scenario_3.pkl
Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc',
       'poa_comp', 'poa_comp_rc', 'poa_rc', 'module_temp', 'cell_temp', 'wind',
       'T', 'albedo', 'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: KUO
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_3//hgbr_pipeline_KUO_scenario_3.pkl
Index(['time', 'ghi', 'dhi', 'dni', 'poa_beam', 'poa_diffused', 'poa_ground',
       'poa_beam_rc', 'poa_dif

In [25]:
scenario = "scenario_4"

helsinki_scenario_4_csv = pick_system_file("Helsinki", folder=f"original/{scenario}", plr="5")
df_hel_scenario_4 = pd.read_csv(helsinki_scenario_4_csv, sep=",")
df_hel_scenario_4 = set_utctime_index(df_hel_scenario_4)
df_hel_scenario_4 = estimate_hgbr(df_hel_scenario_4, config.rated_power_helsinki*1000, "HEL", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc',
       'poa_comp', 'poa_comp_rc', 'poa_rc', 'module_temp', 'cell_temp', 'wind',
       'T', 'albedo', 'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: HEL
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_4//hgbr_pipeline_HEL_scenario_4.pkl


In [26]:
kuopio_scenario_4_csv = pick_system_file("Kuopio", folder=f"original/{scenario}")
df_kuo_scenario_4 = pd.read_csv(kuopio_scenario_4_csv, sep=",")
df_kuo_scenario_4 = set_utctime_index(df_kuo_scenario_4)
df_kuo_scenario_4 = estimate_hgbr(df_kuo_scenario_4, config.rated_power_kuopio*1000, "KUO", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


sodankyla20_scenario_4_csv = pick_system_file("Sodankyla_20", folder=f"original/{scenario}")
df_sod20_scenario_4 = pd.read_csv(sodankyla20_scenario_4_csv, sep=",")
df_sod20_scenario_4 = set_utctime_index(df_sod20_scenario_4)
df_sod20_scenario_4 = estimate_hgbr(df_sod20_scenario_4, config.rated_power_sodankyla*1000, "SOT-20", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


sodankyla90_scenario_4_csv = pick_system_file("Sodankyla_90", folder=f"original/{scenario}")
df_sod90_scenario_4 = pd.read_csv(sodankyla90_scenario_4_csv, sep=",")
df_sod90_scenario_4 = set_utctime_index(df_sod90_scenario_4)
df_sod90_scenario_4 = estimate_hgbr(df_sod90_scenario_4, config.rated_power_sodankyla*1000, "SOT-90", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")


turku_scenario_4_csv = pick_system_file("KTK", folder=f"original/{scenario}")
df_tku_scenario_4 = pd.read_csv(turku_scenario_4_csv, sep=",")
df_tku_scenario_4 = set_utctime_index(df_tku_scenario_4)
df_tku_scenario_4 = estimate_hgbr(df_tku_scenario_4, config.rated_power_turku*1000, "TKU", scenario, "power_without_plr_losses", training_years=1, write_model_path=f"./models/hgbr/{scenario}/")

Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc',
       'poa_comp', 'poa_comp_rc', 'poa_rc', 'module_temp', 'cell_temp', 'wind',
       'T', 'albedo', 'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: KUO
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_4//hgbr_pipeline_KUO_scenario_4.pkl
Index(['time', 'ghi', 'dhi', 'dni', 'poa_beam', 'poa_diffused', 'poa_ground',
       'poa_beam_rc', 'poa_diffused_rc', 'poa_ground_rc', 'poa_comp',
       'poa_comp_rc', 'module_temp', 'cell_temp', 'wind', 'T', 'albedo',
       'power', 'pv_inv_out', 'power_without_plr_losses'],
      dtype='str')

Processing system: SOT-20
Fitting pipeline...
Saving pipeline object to: ./models/hgbr/scenario_4//hgbr_pipeline_SOT-20_scenario_4.pkl
Index(['time', 'ghi', 'dhi', 'dni', 'poa', 'poa_beam', 'poa_diffused',
       'poa_ground', 'poa_beam_rc', 'poa_diffuse